# 02 - Compare candidate LLMs for the RAG baseline

Same retrieved context, same prompt, greedy decoding: only the generator changes. For each model we record
answers, refusals, citation validity, whether "thinking" text leaked, speed and GPU memory. With only the
7 example questions this is a **smoke test for choosing a model, not a benchmark**; the real comparison
happens on the golden set.

Before running: Runtime > T4 GPU, Colab Secret `HF_TOKEN` (notebook access on). Each model downloads
roughly 16-19 GB, so expect several minutes per model.

In [ ]:
import os
import subprocess
import sys

REPO = "agentic-rag-eval"
BRANCH = "p1-agentic-rag"
if not os.path.exists(REPO):
    subprocess.run(
        ["git", "clone", "-b", BRANCH, f"https://github.com/ilaria-cesa/{REPO}.git"], check=True
    )
else:
    subprocess.run(["git", "-C", REPO, "pull"], check=True)
os.chdir(REPO)
sys.path.insert(0, os.getcwd())
%pip install -q -r requirements.txt

In [ ]:
import random

import numpy as np
import torch

assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print(torch.cuda.get_device_name(0))

In [ ]:
from google.colab import userdata
from huggingface_hub import hf_hub_download

from src.eval.golden import load_golden
from src.retrieval.embed import load_chunks

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
HF_REPO = "IlariaCesa/sec-10k-chunks"
chunks = load_chunks(hf_hub_download(HF_REPO, "chunks_1600.jsonl", repo_type="dataset"))
embeddings = np.load(
    hf_hub_download(HF_REPO, "embeddings_bge-m3_1600.npy", repo_type="dataset")
).astype(np.float32)
assert len(chunks) == len(embeddings), "chunks and embeddings are out of sync"
chunk_by_id = {c["chunk_id"]: c for c in chunks}
questions = load_golden("golden/examples.jsonl")
print(len(chunks), "chunks,", len(questions), "questions")

In [ ]:
import gc

from sentence_transformers import SentenceTransformer

from src.retrieval.embed import embed_texts
from src.retrieval.index import build_index, search

K = 5
index = build_index(embeddings)
embedder = SentenceTransformer("BAAI/bge-m3", device="cuda", model_kwargs={"dtype": torch.float16})
embedder.max_seq_length = 640
query_vectors = embed_texts(embedder, [q["question"] for q in questions])
_, top_ids = search(index, query_vectors, k=K)
del embedder
gc.collect()
torch.cuda.empty_cache()

for q, ids in zip(questions, top_ids):
    q["retrieved"] = [chunks[i] for i in ids]
    evidence = set(q["evidence_chunk_ids"])
    hit = len(evidence & {c["chunk_id"] for c in q["retrieved"]})
    print(q["id"], q["type"], f"evidence found in top-{K}: {hit}/{len(evidence)}")

In [ ]:
import time

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

from src.agent.parsing import parse_answer
from src.agent.prompts import build_messages

# Edit this list to change the candidates. Add "Qwen/Qwen2.5-7B-Instruct" for the older generation.
MODELS = ["Qwen/Qwen3-8B", "Qwen/Qwen3.5-9B"]
MAX_NEW_TOKENS = 400

quant = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,  # the T4 has no bfloat16
)


def load_llm(name):
    tokenizer = AutoTokenizer.from_pretrained(name)
    try:
        model = AutoModelForCausalLM.from_pretrained(
            name, quantization_config=quant, device_map="auto", dtype=torch.float16
        )
        loader = "AutoModelForCausalLM"
    except Exception as error:  # noqa: BLE001  some multimodal models are not registered as causal LMs
        print(
            f"AutoModelForCausalLM failed for {name}: {error!r}\nTrying AutoModelForImageTextToText"
        )
        from transformers import AutoModelForImageTextToText

        model = AutoModelForImageTextToText.from_pretrained(
            name, quantization_config=quant, device_map="auto", dtype=torch.float16
        )
        loader = "AutoModelForImageTextToText"
    return tokenizer, model, loader


def generate(tokenizer, model, messages):
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    start = time.time()
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    new_tokens = output[0][inputs["input_ids"].shape[-1] :]
    return (
        tokenizer.decode(new_tokens, skip_special_tokens=True),
        len(new_tokens),
        time.time() - start,
        inputs["input_ids"].shape[-1],
    )

In [ ]:
results = []
for name in MODELS:
    print("=" * 80, "\nLoading", name)
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    tokenizer, model, loader = load_llm(name)
    load_seconds = round(time.time() - t0)
    print(f"loaded with {loader} in {load_seconds}s")
    for q in questions:
        provided = [c["chunk_id"] for c in q["retrieved"]]
        raw, n_tokens, seconds, prompt_tokens = generate(
            tokenizer, model, build_messages(q["question"], q["retrieved"])
        )
        parsed = parse_answer(raw, provided)
        results.append(
            {
                "model": name,
                "id": q["id"],
                "type": q["type"],
                "expected": q["expected"],
                "refused": parsed["refused"],
                "answer": parsed["answer"],
                "cited_ids": parsed["cited_ids"],
                "invalid_citations": parsed["invalid_citations"],
                "had_thinking": parsed["had_thinking"],
                "new_tokens": n_tokens,
                "prompt_tokens": prompt_tokens,
                "seconds": round(seconds, 1),
                "peak_vram_gb": round(torch.cuda.max_memory_allocated() / 1e9, 1),
                "loader": loader,
            }
        )
        print(
            f"\n[{name}] {q['id']} ({q['type']}, expected {q['expected']}) {seconds:.1f}s, {n_tokens} tokens"
        )
        print(parsed["answer"][:600])
    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
import json

import pandas as pd

df = pd.DataFrame(results)
df["correct_behaviour"] = (df["expected"] == "refuse") == df["refused"]
df["no_citation_answer"] = ~df["refused"] & (df["cited_ids"].str.len() == 0)
df["tokens_per_s"] = df["new_tokens"] / df["seconds"]
summary = (
    df.groupby("model")
    .agg(
        correct_answer_vs_refuse=("correct_behaviour", "mean"),
        invalid_citation_answers=("invalid_citations", lambda s: sum(len(x) > 0 for x in s)),
        answers_without_citation=("no_citation_answer", "sum"),
        thinking_leaked=("had_thinking", "sum"),
        mean_seconds=("seconds", "mean"),
        tokens_per_s=("tokens_per_s", "mean"),
        peak_vram_gb=("peak_vram_gb", "max"),
    )
    .round(2)
)
print(summary.to_string())
os.makedirs("results", exist_ok=True)
with open("results/llm_comparison.json", "w") as f:
    json.dump(results, f, indent=1, ensure_ascii=False)
print("\nsaved results/llm_comparison.json")

Copy back to the chat: the summary table, the loader line of each model, and the answers printed above.
Correctness of the answers is judged by reading them against `golden/examples.jsonl`; the table only
measures behaviour (refusals, citations, speed, memory).